## Vector Search

- https://github.com/Azure/azure-search-vector-samples/blob/main/demo-python/code/azure-search-vector-python-sample.ipynb
- https://learn.microsoft.com/en-us/azure/search/vector-search-how-to-query?tabs=query-2023-11-01,filter-2023-11-01

In [ ]:
%pip install azure-search-documents
%pip install azure-core
%pip install pydantic-settings
%pip install requests
%pip install openai
%pip install tiktoken
%pip install tenacitya
%pip install langchain
%pip install pypdf

In [ ]:
from pydantic_settings import BaseSettings

class Env(BaseSettings):
    SEARCH_API_KEY:str
    SEARCH_INDEX:str
    SEARCH_SERVICE_NAME:str
    OPENAI_API_KEY:str
    OPENAI_ENDPOINT:str
    OPENAI_API_VERSION:str = '2023-05-15'
    GPT_35_MODEL_NAME:str = "gpt-35-turbo"
    GPT_4_MODEL_NAME:str = "gpt-4"
    TEXT_EMBEDDING_MODEL_NAME:str="text-embedding-ada-002"
    class Config:
        env_file = "vector-search.env"
        
env = Env()

In [ ]:
from openai import AzureOpenAI
from tenacity import retry, wait_random_exponential, stop_after_attempt  

client = AzureOpenAI(api_key=env.OPENAI_API_KEY, api_version=env.OPENAI_API_VERSION, azure_endpoint=env.OPENAI_ENDPOINT)

@retry(wait=wait_random_exponential(min=1, max=20), stop=stop_after_attempt(6))
# Function to generate embeddings for title and content fields, also used for query embeddings
def generate_embeddings(text, model=env.TEXT_EMBEDDING_MODEL_NAME):
    return client.embeddings.create(input = [text], model=model).data[0].embedding

## Create Index

In [ ]:
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.indexes import SearchIndexClient  
from azure.search.documents.indexes.models import (
    SimpleField,
    SearchableField,
    SearchFieldDataType,
    VectorSearch,
    HnswAlgorithmConfiguration,
    VectorSearchAlgorithmKind,
    HnswParameters,
    VectorSearchAlgorithmMetric,
    ExhaustiveKnnAlgorithmConfiguration,
    ExhaustiveKnnParameters,
    VectorSearchProfile,
    SemanticConfiguration,
    SemanticPrioritizedFields,
    SemanticField,
    SearchIndex,
    SemanticSearch,
    SearchField
)

# Create a search index
fields = [
    SimpleField(
        name="id",
        type=SearchFieldDataType.String,
        key=True,
        sortable=True,
        filterable=True,
    ),
    SearchableField(name="title", type=SearchFieldDataType.String, searchable=True),
    SearchableField(name="content", type=SearchFieldDataType.String, searchable=True),
    SearchField(
        name="title_embedding",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        vector_search_dimensions=1536,
        vector_search_profile_name="myHnswProfile",
    ),
    SearchField(
        name="content_embedding",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        vector_search_dimensions=1536,
        vector_search_profile_name="myHnswProfile",
    ),
]

# Configure the vector search configuration
vector_search = VectorSearch(
    algorithms=[
        HnswAlgorithmConfiguration(
            name="myHnsw",
            kind=VectorSearchAlgorithmKind.HNSW,
            parameters=HnswParameters(
                m=4,
                ef_construction=400,
                ef_search=500,
                metric=VectorSearchAlgorithmMetric.COSINE,
            ),
        ),
        ExhaustiveKnnAlgorithmConfiguration(
            name="myExhaustiveKnn",
            kind=VectorSearchAlgorithmKind.EXHAUSTIVE_KNN,
            parameters=ExhaustiveKnnParameters(
                metric=VectorSearchAlgorithmMetric.COSINE
            ),
        ),
    ],
    profiles=[
        VectorSearchProfile(
            name="myHnswProfile",
            algorithm_configuration_name="myHnsw",
        ),
        VectorSearchProfile(
            name="myExhaustiveKnnProfile",
            algorithm_configuration_name="myExhaustiveKnn",
        ),
    ],
)

semantic_config = SemanticConfiguration(
    name="my-semantic-config",
    prioritized_fields=SemanticPrioritizedFields(
        title_field=SemanticField(field_name="title"),
        # keywords_fields=[SemanticField(field_name="category")],
        content_fields=[SemanticField(field_name="content")]
    )
)

# Create the semantic settings with the configuration
semantic_search = SemanticSearch(configurations=[semantic_config])

# Create the search index with the semantic settings
index = SearchIndex(name=env.SEARCH_INDEX, fields=fields,
                    vector_search=vector_search, semantic_search=semantic_search)

index_client = SearchIndexClient(f"https://{env.SEARCH_SERVICE_NAME}.search.windows.net", AzureKeyCredential(env.SEARCH_API_KEY))
result = index_client.create_or_update_index(index)
print(f' {result.name} created')

## Parse Documents with Langchain

In [23]:
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter


# Read the PDF file
data_path = "../../data/末期腎臟病QA.pdf" # 請換成自己的 PDF 檔案路徑
loader = PyPDFLoader(data_path)
documents = loader.load()

print(f"Number of documents: {len(documents)}")

# Split the document into paragraphs
splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20,
    length_function=len,
    is_separator_regex=False,
)
paragraphs = splitter.split_documents(documents)

print(f"Number of paragraphs: {len(paragraphs)}")

Number of documents: 3
Number of paragraphs: 22


## Prepare Data for Index

In [24]:
from pathlib import Path
import os
import uuid
import json

embedded_paragraphs = []

for paragraph in paragraphs:
    filename = Path(paragraph.metadata.get('source','')).name
    title =  os.path.splitext(filename)[0]
    content = paragraph.page_content
    title_embedding = generate_embeddings(title)
    content_embedding = generate_embeddings(content)
    embedded_paragraphs.append({
        "id": uuid.uuid4().__str__(),
        "title": title,
        "content": content,
        "title_embedding": title_embedding,
        "content_embedding": content_embedding
    })

temp_json_path = "../../temp/docVectors.json"
with open(temp_json_path, "w", encoding="utf-8") as f:
    json.dump(embedded_paragraphs, f)

print("Done")

## Insert Data to Index

### 少量資料

In [25]:
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
import json

file = open(temp_json_path, 'r', encoding="utf-8")
upload_documents = json.load(file)

search_client = SearchClient(
    endpoint=f"https://{env.SEARCH_SERVICE_NAME}.search.windows.net",
    index_name=env.SEARCH_INDEX,
    credential=AzureKeyCredential(env.SEARCH_API_KEY),
)
result = search_client.upload_documents(upload_documents)
print(f"Uploaded {len(upload_documents)} documents") 

Uploaded 22 documents


### 大量資料

In [ ]:
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchIndexingBufferedSender
import json

file = open(temp_json_path, 'r', encoding="utf-8")
upload_documents = json.load(file)

with SearchIndexingBufferedSender(
    endpoint=f"https://{env.SEARCH_SERVICE_NAME}.search.windows.net",
    index_name=env.SEARCH_INDEX,
    credential=AzureKeyCredential(env.SEARCH_API_KEY),
) as batch_client:
    batch_client.upload_documents(documents=upload_documents)  
print(f"Uploaded {len(upload_documents)} documents in total")  

## Perform a vector similarity search

In [26]:
from azure.search.documents import SearchClient
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.models import VectorizedQuery


query = "末期腎臟病是什麼？" 

search_client = SearchClient(f"https://{env.SEARCH_SERVICE_NAME}.search.windows.net", env.SEARCH_INDEX, credential=AzureKeyCredential(env.SEARCH_API_KEY))
vector_query = VectorizedQuery(vector=generate_embeddings(query), k_nearest_neighbors=3, fields="content_embedding")

results = search_client.search(  
    search_text=None,  
    vector_queries= [vector_query],
    select=["title", "content"],
)  

for result in results:
    print(f"{result['title']}")  
    print(f"{result['content']}")  
    print(f"Score: {result['@search.score']}\n")

末期腎臟病QA
末期腎臟病 ，我應該選擇哪一種治療方式 ? 
當醫療人員告訴我，可能要開始考慮洗腎或換腎  
Q：我的腎臟功能有可能恢復成比現在還好的狀態嗎？
Score: 0.90520585

末期腎臟病QA
情境模擬影片： 面對末期腎臟病，我應該選擇哪一種治療方式  
歡迎使用手機掃瞄 QR Code 觀看影片  
國語版  台語版  客語版
Score: 0.88934463

末期腎臟病QA
A：如果是急性腎衰竭是可以的；若是慢性腎 臟病就要依靠低蛋白飲食控
制來降低尿毒素，但無法恢復到正常 。 
Q：有可能晚一點洗腎或不洗腎嗎？如果不洗腎，會怎麼樣？
Score: 0.8806243



## Perform an Exhaustive KNN exact nearest neighbor search

In [28]:
from azure.search.documents import SearchClient
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.models import VectorizedQuery


query = "末期腎臟病是什麼？" 

search_client = SearchClient(f"https://{env.SEARCH_SERVICE_NAME}.search.windows.net", env.SEARCH_INDEX, credential=AzureKeyCredential(env.SEARCH_API_KEY))
vector_query = VectorizedQuery(vector=generate_embeddings(query), k_nearest_neighbors=3, fields="content_embedding", exhaustive=True)

results = search_client.search(  
    search_text=None,  
    vector_queries= [vector_query],
    select=["title", "content"],
)  

for result in results:
    print(f"{result['title']}")  
    print(f"{result['content']}")  
    print(f"Score: {result['@search.score']}\n")

末期腎臟病QA
末期腎臟病 ，我應該選擇哪一種治療方式 ? 
當醫療人員告訴我，可能要開始考慮洗腎或換腎  
Q：我的腎臟功能有可能恢復成比現在還好的狀態嗎？
Score: 0.90520585

末期腎臟病QA
情境模擬影片： 面對末期腎臟病，我應該選擇哪一種治療方式  
歡迎使用手機掃瞄 QR Code 觀看影片  
國語版  台語版  客語版
Score: 0.88934445

末期腎臟病QA
A：如果是急性腎衰竭是可以的；若是慢性腎 臟病就要依靠低蛋白飲食控
制來降低尿毒素，但無法恢復到正常 。 
Q：有可能晚一點洗腎或不洗腎嗎？如果不洗腎，會怎麼樣？
Score: 0.8806239



## Perform a Cross-Field Vector Search

多欄位的向量搜尋

In [27]:
from azure.search.documents import SearchClient
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.models import VectorizedQuery


query = "末期腎臟病是什麼？" 

search_client = SearchClient(f"https://{env.SEARCH_SERVICE_NAME}.search.windows.net", env.SEARCH_INDEX, credential=AzureKeyCredential(env.SEARCH_API_KEY))

# 用 embedded query 搜尋 title_embedding 和 content_embedding 兩個欄位
vector_query = VectorizedQuery(vector=generate_embeddings(query), k_nearest_neighbors=3, fields="title_embedding, content_embedding")

results = search_client.search(  
    search_text=None,  
    vector_queries= [vector_query],
    select=["title", "content"],
)  

for result in results:
    print(f"{result['title']}")  
    print(f"{result['content']}")  
    print(f"Score: {result['@search.score']}\n")

末期腎臟病QA
末期腎臟病 ，我應該選擇哪一種治療方式 ? 
當醫療人員告訴我，可能要開始考慮洗腎或換腎  
Q：我的腎臟功能有可能恢復成比現在還好的狀態嗎？
Score: 0.01666666753590107

末期腎臟病QA
A：有圖片或影片 可以看。 
Q：洗腎會影響到旅遊或出差嗎？  
A：可以事先安排聯絡透析中心 。 
Q：洗腎會影響到生育能力嗎？
Score: 0.01666666753590107

末期腎臟病QA
情境模擬影片： 面對末期腎臟病，我應該選擇哪一種治療方式  
歡迎使用手機掃瞄 QR Code 觀看影片  
國語版  台語版  客語版
Score: 0.016393441706895828

末期腎臟病QA
B. 關於換腎，我想了解：  
Q：換腎之後，新的腎臟可能可以用多久？如果不能用了，會怎麼樣？  
A：因人而異 。如果不能用 ，仍可以 保有第 2次移植機會或 改用透析治療。
Score: 0.016393441706895828

末期腎臟病QA
Q：如果我的家人捐腎給我，以後他洗腎的機率會不會變高？  
A：移植前醫師會對捐贈 者做詳細的評估 ，確認捐贈者風險低於 1%，才會
同意移植 。
Score: 0.016129031777381897

末期腎臟病QA
A：如果是急性腎衰竭是可以的；若是慢性腎 臟病就要依靠低蛋白飲食控
制來降低尿毒素，但無法恢復到正常 。 
Q：有可能晚一點洗腎或不洗腎嗎？如果不洗腎，會怎麼樣？
Score: 0.016129031777381897

